In the bronze layer, this dataset had the latitude and longitude of each healthcare facility in California. We need to map them in the MSSA database

In [0]:
import pandas as pd

hpsa_ca_site_dashboard = "ca_healthcare_fac_bronze.hpsa_site_dashboards.hpsa_ca_site_dashboard"
hpsa_ca_sites_df = spark.read.table(hpsa_ca_site_dashboard).toPandas()
hpsa_ca_sites_df.head()

In [0]:
from shapely.geometry import Point
import geopandas as gpd

hpsa_dashboard_gpd = gpd.GeoDataFrame(hpsa_ca_sites_df, geometry=gpd.points_from_xy(hpsa_ca_sites_df.longitude, hpsa_ca_sites_df.latitude), crs="EPSG:4326")
hpsa_dashboard_gpd.head()

In [0]:
hpsa_ca_sites_df.shape

In [0]:
%pip install geopandas

In [0]:
mssa_bronze_pddf = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
print(mssa_bronze_pddf.columns)

In [0]:
from shapely.geometry import Polygon

def convert_polygon(rings):
    return Polygon(rings[0])

mssa_bronze_pddf['geometry_rings'] = mssa_bronze_pddf['geometry_rings'].apply(convert_polygon)
mssa_bronze_pddf.head()


In [0]:
mssa_geo_gdf = gpd.GeoDataFrame(mssa_bronze_pddf[['TRACTCE', 'GEOID', 'MSSAID', 'MSSANM', 'geometry_rings']], geometry='geometry_rings', crs='EPSG:4326')
mssa_geo_gdf.head()


In [0]:
hpsa_ca_dashboard_mssa = hpsa_dashboard_gpd.sjoin_nearest(mssa_geo_gdf, how='left')


In [0]:
hpsa_ca_dashboard_mssa.head()

36 surrrogate keys merge to 2 mssaid - we will keep unique values only

In [0]:
hpsa_ca_dashboard_mssa = hpsa_ca_dashboard_mssa.drop_duplicates(subset=['site_dashboard_surrogate_key'])
hpsa_ca_dashboard_mssa = hpsa_ca_dashboard_mssa.drop("geometry", axis=1)
hpsa_ca_dashboard_mssa.shape


In [0]:
hpsa_ca_dashboard_mssa_df = spark.createDataFrame(hpsa_ca_dashboard_mssa)
hpsa_ca_dashboard_mssa_df.write.mode("overwrite").saveAsTable("ca_healthcare_fac_silver.hpsa_dashboard_silver.hpsa_ca_dashboard_mssa")